# Quoridor AI — POC Training v3.2 (5×5)

**Group 501** | Colman College | DL Final Project

AlphaZero-inspired agent for Quoridor trained via self-play.

### Fixes in v3.2
* Increased dirichlet alpha from 0.3 to 1.2 to increase exploration

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells in order
3. If Colab disconnects, re-run from **Section 1** — `resume=True` picks up automatically

---
## 1. Environment Setup
Run once per Colab session.

In [ ]:
import os
import sys

REPO_DIR = "./dl-quoridor"
BRANCH = "fixes"

# 1. Clone the repository to get the 'src' folder
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ReefKenig/dl-quoridor.git {REPO_DIR}

# 2. Move into the downloaded project folder
os.chdir(REPO_DIR)

# 3. Checkout the branch that includes parallel/batched inference fixes
!git fetch --all
!git checkout {BRANCH}
!git pull

# 4. Install dependencies to the active Jupyter kernel (using %pip instead of !pip)
%pip install -r requirements.txt

# 5. Ensure Python knows where to find the 'src' module
sys.path.append(os.getcwd())
print("Setup complete!")

Fetching origin
Already on 'fixes'
Your branch is up to date with 'origin/fixes'.
Already up to date.
  Using cached wandb-0.27.2-py3-none-manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached pytest-9.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached flask-3.0.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached stable_baselines3-2.9.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached gymnasium-1.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached shimmy-2.0.1-py3-none-any.whl.metadata (4.4 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached gitpython-3.1.50-py3-none-any.whl.metadata (14 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
Using cached flask-3.0.2-py3-none-any.whl (101 kB)
Using cached wandb-0.27.2-py3-none-manylinux_2_28_x86_64.whl (26.4 MB)
Using cached pydantic-2.13.4-py3-none-a

In [2]:
# 0 — Timestamp & duration utilities (run first)
!pip install humanize -q

import time
from datetime import datetime
import humanize

_cell_timers = {}

def ts(label=""):
    """Print a timestamp and start a timer for this label."""
    now = datetime.now()
    _cell_timers[label] = time.monotonic()
    tag = f"[{label}] " if label else ""
    print(f"⏱ {tag}Started at {now.strftime('%Y-%m-%d %H:%M:%S')}")

def te(label=""):
    """Print elapsed duration since ts() was called with the same label."""
    now = datetime.now()
    elapsed = time.monotonic() - _cell_timers.get(label, time.monotonic())
    tag = f"[{label}] " if label else ""
    duration = humanize.precisedelta(elapsed, minimum_unit="seconds", format="%0.0f")
    print(f"✅ {tag}Finished at {now.strftime('%Y-%m-%d %H:%M:%S')} — took {duration}")

print("Timestamp utils loaded (ts/te)")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Timestamp utils loaded (ts/te)


In [3]:
import sys
import subprocess

print(f"Installing PyTorch to: {sys.executable}\n")

# Force install torch directly to this kernel, ignoring cache to prevent disk-space crashes
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "torch", "numpy", "--no-cache-dir"],
    capture_output=True,
    text=True
)

print("--- INSTALLATION OUTPUT ---")
print(result.stdout)

if result.stderr:
    print("--- ERRORS / WARNINGS ---")
    print(result.stderr)
else:
    print("\n✅ Installation completed with no errors.")

Installing PyTorch to: /usr/bin/python3

--- INSTALLATION OUTPUT ---

--- ERRORS / WARNINGS ---

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip



In [4]:
# 1.1 — Verify GPU
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be slow.")
    print("Go to Runtime → Change runtime type → T4 GPU")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Memory: 21.6 GB


In [5]:
# ── Anti-disconnect for Colab ──
# This JavaScript pings the Colab runtime to prevent idle timeout
# import IPython
# IPython.display.display(IPython.display.Javascript('''
# function KeepAlive() {
#     console.log("Keeping alive " + new Date().toLocaleTimeString());
#     document.querySelector("colab-toolbar-button#connect")?.click();
# }
# setInterval(KeepAlive, 60000);
# '''))
# print("Anti-disconnect activated (pings every 60s)")


In [6]:
# 1.2 — Mount Google Drive (checkpoints & logs persist here)
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)
DRIVE_ROOT = "./v3_2"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints"
LOG_DIR = f"{DRIVE_ROOT}/logs"

!mkdir -p "{CHECKPOINT_DIR}" "{LOG_DIR}"
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Logs:        {LOG_DIR}")

Checkpoints: ./v3_2/checkpoints
Logs:        ./v3_2/logs


---
## 2. Validation
Confirm all components work before training.

In [7]:
ts("Smoke test")

# 2.1 — Smoke test: full pipeline (env → tensor → model → MCTS → self-play)
from src.env.quoridor_env import QuoridorEnv
from src.model.network import QuoridorModel
from src.mcts.mcts import MCTS, MCTSConfig
from src.mcts.self_play import play_one_game

env = QuoridorEnv(board_size=5, max_walls_per_player=3)
model = QuoridorModel(
    board_size=5,
    action_space_size=44,
    in_channels=11,
    num_channels=16,
    num_res_blocks=2
)

def nn_evaluate(state):
    tensor = env.state_to_tensor(state)
    return model.predict(tensor)

mcts = MCTS(config=MCTSConfig(num_simulations=50), evaluate_fn=nn_evaluate)
samples, winner = play_one_game(env, mcts, max_moves=100)

original_count = len(samples) // 2  # half are augmented mirrors
print(f"Game: {original_count} moves, winner=P{winner}")
print(f"Total samples (with augmentation): {len(samples)} ({len(samples)/original_count:.0f}x)")
print(f"Tensor shape: {samples[0].state.shape}")
print(f"Policy shape: {samples[0].policy_target.shape}")

# Verify discounted values are NOT flat
values = [s.value_target for s in samples[:original_count]]
unique_vals = len(set(round(v, 4) for v in values))
print(f"Unique value targets: {unique_vals} (should be > 2)")

assert samples[0].state.shape == (5, 5, 11)
assert samples[0].policy_target.shape == (44,)
assert unique_vals > 2, "Value targets should be discounted, not flat!"
print("\n✓ Smoke test passed — pipeline + augmentation + discounted values working.")

te("Smoke test")

⏱ [Smoke test] Started at 2026-06-17 14:31:06
Game: 50 moves, winner=P1
Total samples (with augmentation): 100 (2x)
Tensor shape: (5, 5, 11)
Policy shape: (44,)
Unique value targets: 50 (should be > 2)

✓ Smoke test passed — pipeline + augmentation + discounted values working.
✅ [Smoke test] Finished at 2026-06-17 14:31:09 — took 4 seconds


In [8]:
# 2.2 — MCTS (random rollouts) vs random agent on real env
from src.mcts.evaluator import evaluate_against_random, mcts_agent

env = QuoridorEnv(board_size=5, max_walls_per_player=3)
mcts_random = MCTS(config=MCTSConfig(num_simulations=200))
agent = mcts_agent(mcts_random, temperature=0.1)

result = evaluate_against_random(env, agent, num_games=20)
print(result.summary())
print(f"\nMCTS wins {result.agent_a_win_rate:.0%} vs random (expected >60%)")

Games: 20 | A wins: 20 (100.0%) | B wins: 0 (0.0%) | Draws: 0 | Avg moves: 21.3 | Avg time: 1.65s

MCTS wins 100% vs random (expected >60%)


---
## 3. Training

In [ ]:
ts("Training")

# 3 — Full POC training using BATCHED GPU inference (parallel self-play)
# This replaces the single-process training_loop with the parallel path
# that actually uses model.predict_batch for GPU-batched MCTS leaf evaluation.
#
# Re-run this cell after Colab disconnects — resume=True picks up from last checkpoint

import copy
import logging
import os
import time
import numpy as np

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    force=True,
)
logger = logging.getLogger("parallel_training")

from src.env.quoridor_env import QuoridorEnv
from src.model.network import QuoridorModel
from src.mcts.mcts import MCTS, MCTSConfig
from src.mcts.self_play import TrainingConfig, ReplayBuffer, TrainingSample, _augment_sample
from src.mcts.parallel_self_play import generate_parallel_self_play_data
from src.mcts.evaluator import evaluate, evaluate_against_random, mcts_agent, random_agent
from src.utils.config import load_config
from src.utils.checkpoint import CheckpointManager
from src.utils.logger import TrainingLogger

print("USING_BATCHED_INFERENCE=True — parallel self-play with predict_batch")

# --- Config ---
cfg = load_config("configs/config_5x5.json")

env = QuoridorEnv(board_size=5, max_walls_per_player=3)

model = QuoridorModel(
    board_size=5,
    action_space_size=44,
    in_channels=11,
    num_channels=64,
    num_res_blocks=4,
    lr=0.001
)

config = TrainingConfig(
    num_iterations=40,
    games_per_iteration=200,
    mcts_simulations=150,
    batch_size=256,
    training_epochs=8,
    eval_games=20,
    eval_random_games=20,
    win_threshold=0.55,
    replay_buffer_size=30000,
    self_play_checkpoint_freq=20,
    max_game_moves=60,
)

# Parallel self-play settings
NUM_WORKERS = 32
GAMES_PER_WORKER = config.games_per_iteration // NUM_WORKERS  # 200/32 = 6 each

# --- Setup ---
buffer = ReplayBuffer(max_size=config.replay_buffer_size)
ckpt = CheckpointManager(base_dir=CHECKPOINT_DIR, keep_last_n=3)
train_logger = TrainingLogger(
    project="quoridor-ai",
    run_name="v3_2_parallel",
    log_dir=LOG_DIR,
    use_wandb=False,
    config={},
)

best_model = copy.deepcopy(model)
start_iteration = 0

# --- Resume ---
loaded_state = ckpt.load_latest(replay_buffer_max_size=config.replay_buffer_size)
if loaded_state is not None:
    start_iteration = loaded_state["iteration"]
    model.load(loaded_state["model_path"])
    buffer = loaded_state["replay_buffer"]
    best_model_path = loaded_state.get("best_model_path", "")
    if best_model_path and os.path.exists(best_model_path):
        best_model.load(best_model_path)
    logger.info("Resumed from iteration %d, buffer=%d", start_iteration, len(buffer))

# --- Training Loop (parallel self-play) ---
CHECKPOINT_FREQ = 20  # save mid-iteration checkpoint every N games

for iteration in range(start_iteration, config.num_iterations):
    logger.info("=" * 50)
    logger.info("Iteration %d/%d", iteration + 1, config.num_iterations)

    # 1. PARALLEL SELF-PLAY (uses predict_batch on GPU)
    logger.info("  Generating %d self-play games (parallel, %d workers)...",
                config.games_per_iteration, NUM_WORKERS)
    t0 = time.time()

    # Collect samples incrementally with progress + mid-iteration checkpoints
    iteration_samples = []

    def on_games_complete(games_done, total_games, new_samples):
        # Augment and add to buffer
        for state_hwc, policy, value in new_samples:
            sample = TrainingSample(state=state_hwc, policy_target=policy, value_target=value)
            iteration_samples.append(sample)
            aug = _augment_sample(sample, env.board_size, env.action_space_size)
            iteration_samples.append(aug)

        # Progress log
        if games_done % 20 == 0 or games_done == total_games:
            logger.info("    Games: %d/%d", games_done, total_games)

        # Mid-iteration checkpoint
        if games_done % CHECKPOINT_FREQ == 0 and games_done < total_games:
            buffer.add(iteration_samples[:])  # add current samples
            ckpt.save(
                iteration=iteration,
                model=model,
                replay_buffer=buffer,
                metrics={"self_play_game": games_done},
                is_best=False,
            )
            logger.info("    Mid-iteration checkpoint saved (%d/%d games)", games_done, total_games)

    raw_samples = generate_parallel_self_play_data(
        model=model,
        config=cfg,
        num_workers=NUM_WORKERS,
        games_per_worker=GAMES_PER_WORKER,
        batch_size=64,
        on_games_complete=on_games_complete,
    )

    sp_duration = time.time() - t0

    # Add any remaining samples not yet in buffer from augmentation in callback
    # (callback already built iteration_samples, so just add to buffer)
    buffer.add(iteration_samples)
    logger.info("  Self-play: %d samples (%.1fs) | Buffer: %d",
                len(iteration_samples), sp_duration, len(buffer))

    # 2. TRAIN
    if len(buffer) < config.batch_size:
        logger.info("  Buffer too small, skipping training.")
        continue

    logger.info("  Training for %d epochs...", config.training_epochs)
    steps_per_epoch = max(1, len(buffer) // config.batch_size)
    total_steps = steps_per_epoch * config.training_epochs
    total_lp, total_lv = 0.0, 0.0

    t0 = time.time()
    for _ in range(total_steps):
        states, policies, values = buffer.sample_batch(config.batch_size)
        lp, lv = model.train_step(states, policies, values)
        total_lp += lp
        total_lv += lv
    train_duration = time.time() - t0

    avg_lp = total_lp / total_steps
    avg_lv = total_lv / total_steps
    logger.info("  Losses — policy: %.4f, value: %.4f (%.1fs)", avg_lp, avg_lv, train_duration)

    # 3. EVAL vs best
    def nn_eval(state):
        return model.predict(env.state_to_tensor(state))
    def best_nn_eval(state):
        return best_model.predict(env.state_to_tensor(state))

    mcts_new = MCTS(config=MCTSConfig(num_simulations=config.mcts_simulations), evaluate_fn=nn_eval)
    mcts_best = MCTS(config=MCTSConfig(num_simulations=config.mcts_simulations), evaluate_fn=best_nn_eval)

    t0 = time.time()
    result_vs_best = evaluate(env, agent_a=mcts_agent(mcts_new, 0.1),
                              agent_b=mcts_agent(mcts_best, 0.1),
                              num_games=config.eval_games, verbose=True)
    eval_duration = time.time() - t0

    is_best = result_vs_best.should_accept(threshold=config.win_threshold)
    if is_best:
        best_model.copy_weights_from(model)
        logger.info("  Model ACCEPTED (%.0f%%)", result_vs_best.agent_a_win_rate * 100)
    else:
        logger.info("  Model REJECTED (%.0f%%)", result_vs_best.agent_a_win_rate * 100)

    # 4. EVAL vs random
    result_vs_random = evaluate(env, agent_a=mcts_agent(mcts_new, 0.1),
                                agent_b=random_agent(),
                                num_games=config.eval_random_games, verbose=False)
    logger.info("  Win rate vs random: %.0f%%", result_vs_random.agent_a_win_rate * 100)

    # 5. CHECKPOINT
    ckpt.save(iteration=iteration + 1, model=model, replay_buffer=buffer,
              metrics={"loss_policy": avg_lp, "loss_value": avg_lv},
              is_best=is_best)

    # 6. LOG
    train_logger.log_iteration(
        iteration=iteration + 1,
        loss_policy=avg_lp,
        loss_value=avg_lv,
        win_rate_vs_random=result_vs_random.agent_a_win_rate,
        win_rate_vs_best=result_vs_best.agent_a_win_rate,
        model_accepted=is_best,
        avg_game_length=result_vs_random.avg_game_length,
        buffer_size=len(buffer),
        samples_generated=len(iteration_samples),
        self_play_duration_s=sp_duration,
        training_duration_s=train_duration,
        eval_duration_s=eval_duration,
        total_games_played=config.games_per_iteration,
    )

logger.info("Training complete!")
te("Training")

2026-06-17 14:31:43,024 [src.model.network] INFO: QuoridorModel: 306571 params, device=cuda, board=5x5, actions=44
2026-06-17 14:31:43,025 [src.utils.checkpoint] INFO: No checkpoint found in v3_2/checkpoints
2026-06-17 14:31:43,034 [src.mcts.self_play] INFO: ==================================================
2026-06-17 14:31:43,035 [src.mcts.self_play] INFO: Iteration 1/40
2026-06-17 14:31:43,036 [src.mcts.self_play] INFO:   Generating 200 self-play games...


⏱ [Training] Started at 2026-06-17 14:31:43


2026-06-17 14:33:19,837 [src.mcts.self_play] INFO:     Games: 20/200
2026-06-17 14:33:19,846 [src.model.network] INFO: Model saved to v3_2/checkpoints/iter_0000/model.pt
2026-06-17 14:33:19,852 [src.utils.checkpoint] INFO: Checkpoint saved: iter=0, buffer=1018, path=v3_2/checkpoints/iter_0000
2026-06-17 14:33:19,853 [src.mcts.self_play] INFO:     Mid-iteration checkpoint saved (20/200 games)
2026-06-17 14:34:52,824 [src.mcts.self_play] INFO:     Games: 40/200
2026-06-17 14:34:52,845 [src.model.network] INFO: Model saved to v3_2/checkpoints/iter_0000/model.pt
2026-06-17 14:34:52,860 [src.utils.checkpoint] INFO: Checkpoint saved: iter=0, buffer=2050, path=v3_2/checkpoints/iter_0000
2026-06-17 14:34:52,860 [src.mcts.self_play] INFO:     Mid-iteration checkpoint saved (40/200 games)
2026-06-17 14:36:42,198 [src.mcts.self_play] INFO:     Games: 60/200
2026-06-17 14:36:42,208 [src.model.network] INFO: Model saved to v3_2/checkpoints/iter_0000/model.pt
2026-06-17 14:36:42,231 [src.utils.check

---
## 4. Monitor & Analyze Results

In [ ]:
# 4.1 — Training curves
import json
import matplotlib.pyplot as plt

log_path = f"{LOG_DIR}/metrics_full.json"

try:
    with open(log_path) as f:
        metrics = json.load(f)
except FileNotFoundError:
    print("No metrics yet — run training first.")
    metrics = []

if metrics:
    iters = [m['iteration'] for m in metrics]
    loss_p = [m['loss_policy'] for m in metrics]
    loss_v = [m['loss_value'] for m in metrics]
    wr = [m['win_rate_vs_random'] for m in metrics]
    avg_len = [m['avg_game_length'] for m in metrics]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Quoridor 5×5 POC v2 — Training Progress', fontsize=14)

    axes[0, 0].plot(iters, loss_p, 'b-o', markersize=3)
    axes[0, 0].set_title('Policy Loss')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Cross-Entropy')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(iters, loss_v, 'r-o', markersize=3)
    axes[0, 1].set_title('Value Loss')
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].set_ylabel('MSE')
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(iters, [w * 100 for w in wr], 'g-o', markersize=3)
    axes[1, 0].axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
    axes[1, 0].set_title('Win Rate vs Random Agent')
    axes[1, 0].set_xlabel('Iteration')
    axes[1, 0].set_ylabel('Win Rate (%)')
    axes[1, 0].set_ylim(0, 105)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(iters, avg_len, 'm-o', markersize=3)
    axes[1, 1].set_title('Average Game Length')
    axes[1, 1].set_xlabel('Iteration')
    axes[1, 1].set_ylabel('Moves')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{LOG_DIR}/training_curves.png", dpi=150)
    plt.show()

    print(f"\nLatest iteration: {iters[-1]}")
    print(f"Policy loss: {loss_p[-1]:.4f}")
    print(f"Value loss:  {loss_v[-1]:.4f}")
    print(f"Win rate:    {wr[-1]:.1%}")

In [ ]:
# 4.2 — Metrics table
if metrics:
    print(f"{'Iter':>4} | {'Loss_P':>8} | {'Loss_V':>8} | {'WR_Random':>10} | {'Avg_Len':>8} | {'Best':>5}")
    print("-" * 60)
    for m in metrics:
        print(
            f"{m['iteration']:4d} | "
            f"{m['loss_policy']:8.4f} | "
            f"{m['loss_value']:8.4f} | "
            f"{m['win_rate_vs_random']:9.1%} | "
            f"{m['avg_game_length']:8.1f} | "
            f"{'*' if m.get('model_accepted') else ''}"
        )

---
## 5. Evaluate Best Model

In [ ]:
# 5.1 — Load best model and evaluate thoroughly
from src.env.quoridor_env import QuoridorEnv
from src.model.network import QuoridorModel
from src.mcts.mcts import MCTS, MCTSConfig
from src.mcts.evaluator import evaluate_against_random, mcts_agent
import os

env = QuoridorEnv(board_size=5, max_walls_per_player=3)
best_model = QuoridorModel(
    board_size=5,
    action_space_size=44,
    in_channels=11
)

best_path = f"{CHECKPOINT_DIR}/best/model.pt"
if os.path.exists(best_path):
    best_model.load(best_path)
    print(f"Loaded best model from {best_path}")
else:
    print("No best model found — run training first.")

def best_nn_evaluate(state):
    tensor = env.state_to_tensor(state)
    return best_model.predict(tensor)

# Evaluate at different MCTS simulation counts
for sims in [100, 200, 400]:
    mcts = MCTS(
        config=MCTSConfig(num_simulations=sims),
        evaluate_fn=best_nn_evaluate,
    )
    agent = mcts_agent(mcts, temperature=0.1)
    result = evaluate_against_random(env, agent, num_games=50)
    print(f"  {sims:>4} sims: {result.summary()}")

In [ ]:
# 5.2 — Compare: trained model vs untrained model
from src.mcts.evaluator import evaluate

untrained_model = QuoridorModel(
    board_size=5,
    action_space_size=44,
    in_channels=11
)

def untrained_evaluate(state):
    tensor = env.state_to_tensor(state)
    return untrained_model.predict(tensor)

mcts_trained = MCTS(
    config=MCTSConfig(num_simulations=200),
    evaluate_fn=best_nn_evaluate,
)
mcts_untrained = MCTS(
    config=MCTSConfig(num_simulations=200),
    evaluate_fn=untrained_evaluate,
)

trained_agent = mcts_agent(mcts_trained, temperature=0.1)
untrained_agent = mcts_agent(mcts_untrained, temperature=0.1)

result = evaluate(env, agent_a=trained_agent, agent_b=untrained_agent, num_games=40)
print(f"Trained vs Untrained: {result.summary()}")